# 06 大促复盘（11-15 ~ 11-17）

**业务问题**：大促带来了哪些用户？谁在购买？带来的用户留下了吗？是否透支了后续需求？

**核心结论**

- **大促对早期风险用户的拉动最明显**，到访率是平时的 1.65 倍；增量访客约六成来自健康用户。
- 大促日各层用户的下单转化率都提升了 3–5 倍。GMV 增量约 45% 来自健康用户，约四分之一来自大促期间的新访客。
- **大促带回的老用户留存弱于平时**（7 天内再访低 7–13 个百分点），召回效果是一次性的。建议在大促后第 6–8 天对大促访客补充触达。
- 大促后一周 GMV 没有下降，未见需求透支。

数据中没有活动日历，大促窗口根据流量识别。大促前两天恰好遇上购买数据采集中断（见报告 01），本节只使用不受影响的指标。

In [1]:
from nbutils import *
import numpy as np

d = q("SELECT * FROM ads_daily_metrics ORDER BY event_date")
d["event_date"] = pd.to_datetime(d.event_date)

## 1. 大促窗口识别与数据可用性

In [2]:
d["dau_vs_lastweek"] = d.dau / d.dau.shift(7)
d["views_vs_lastweek"] = d.views / d.views.shift(7)
w = d[(d.event_date >= "2019-11-12") & (d.event_date <= "2019-11-20")][["event_date", "dau", "dau_vs_lastweek", "views_vs_lastweek", "orders", "gmv"]].copy()
w["购买数据"] = np.select([w.event_date == "2019-11-15", w.event_date == "2019-11-16"], ["缺失（全天）", "缺失（约 16 小时）"], "完整")
w.columns = ["日期", "DAU", "DAU 比上周同日", "浏览量比上周同日", "订单数", "GMV", "购买数据"]
w.style.format({"日期": "{:%m-%d %a}", "DAU": "{:,}", "DAU 比上周同日": "{:.2f}×", "浏览量比上周同日": "{:.2f}×",
                "订单数": "{:,.0f}", "GMV": "{:,.0f}"}).hide(axis="index")

日期,DAU,DAU 比上周同日,浏览量比上周同日,订单数,GMV,购买数据
11-12 Tue,"285,987",1.09×,1.13×,"19,347","6,793,536",完整
11-13 Wed,"285,549",1.11×,1.17×,"19,117","6,815,581",完整
11-14 Thu,"304,003",1.09×,1.54×,"19,020","7,088,729",完整
11-15 Fri,"380,298",1.33×,3.24×,0,0,缺失（全天）
11-16 Sat,"489,087",1.82×,3.41×,"56,659","23,443,620",缺失（约 16 小时）
11-17 Sun,"489,392",1.77×,3.17×,"150,911","57,517,324",完整
11-18 Mon,"320,560",1.10×,1.01×,"24,193","8,341,939",完整
11-19 Tue,"291,031",1.02×,0.86×,"21,063","7,310,335",完整
11-20 Wed,"292,102",1.02×,0.83×,"21,149","7,062,797",完整


11-15 ~ 11-17 三天 DAU 是上周同日的 1.3–1.8 倍、浏览量 3.2–3.4 倍，11-18 起立刻回落，大促窗口就定为这三天（11-14 晚上浏览量已经开始抬升，是预热）。

| 能用 | 不能用 |
|---|---|
| 三天的流量、到访、加购 | 11-15 全天、11-16 约 16 小时的购买和 GMV |
| 11-17 全天的购买（只缺本地 0–6 点） | 三天合计 GMV（只能当下限） |
| 大促前后的一切指标 | 「大促前段 vs 后段」的转化对比 |

若未识别采集中断，会从数据中得出「大促前两天用户只加购不下单，最后一天集中清空购物车」的判断——
这一判断完全由数据缺失导致，不能作为结论。

## 2. 到访拉动：分层对比

以大促前一天（11-14）的生命周期分层为准，看每一层有多少人在大促三天里来过；
对照组取两周前同样是周五到周日的三天（11-01 ~ 11-03，分层快照日 10-31）。
长期沉默层在 10-31 的快照里还不存在（需要 30 天以上历史），所以不参与对比。

In [3]:
def window_stats(snap, w0, w1, r0, r1):
    return q(f"""
        WITH st AS (
            SELECT v.user_id,
                   lifecycle_stage(DATE '{snap}' - v.first_seen_date, DATE '{snap}' - v.event_date) AS stage
            FROM dws_user_visit_interval v
            WHERE v.event_date <= DATE '{snap}'
              AND (v.next_active_date IS NULL OR v.next_active_date > DATE '{snap}')
        ),
        w AS (SELECT DISTINCT user_id FROM dws_user_daily WHERE event_date BETWEEN '{w0}' AND '{w1}'),
        r AS (SELECT DISTINCT user_id FROM dws_user_daily WHERE event_date BETWEEN '{r0}' AND '{r1}')
        SELECT st.stage, COUNT(*) AS users, COUNT(w.user_id) AS visited,
               COUNT(r.user_id) FILTER (WHERE w.user_id IS NOT NULL) AS visited_and_returned
        FROM st LEFT JOIN w USING (user_id) LEFT JOIN r USING (user_id)
        WHERE st.stage <> 'long_lapsed'
        GROUP BY 1
    """).set_index("stage").loc[["new", "healthy", "early_risk", "accelerating"]]

camp = window_stats("2019-11-14", "2019-11-15", "2019-11-17", "2019-11-18", "2019-11-24")
base = window_stats("2019-10-31", "2019-11-01", "2019-11-03", "2019-11-04", "2019-11-10")
reach = pd.DataFrame({
    "分层人数（大促前一天）": camp.users,
    "大促三天到访率": camp.visited / camp.users,
    "对照三天到访率": base.visited / base.users,
})
reach["相对提升"] = reach.大促三天到访率 / reach.对照三天到访率
reach["增量到访人数"] = camp.users * (reach.大促三天到访率 - reach.对照三天到访率)
reach["占增量的比例"] = reach.增量到访人数 / reach.增量到访人数.sum()
reach.index = reach.index.map(STAGE_CN)
reach.style.format({"分层人数（大促前一天）": "{:,}", "大促三天到访率": "{:.1%}", "对照三天到访率": "{:.1%}",
                    "相对提升": "{:.2f}×", "增量到访人数": "{:,.0f}", "占增量的比例": "{:.0%}"})

,分层人数（大促前一天）,大促三天到访率,对照三天到访率,相对提升,增量到访人数,占增量的比例
stage,,,,,,
新用户,"590,360",14.2%,11.9%,1.19×,"13,452",6%
健康,"923,498",40.8%,27.4%,1.49×,"124,288",59%
早期风险,"707,135",16.1%,9.8%,1.65×,"44,999",21%
加速流失,"1,148,797",8.3%,5.9%,1.41×,"28,056",13%


- **增量到访的绝对量主要来自健康层**（约六成），因为这一层人多、本来就活跃；
- **相对拉动最大的是早期风险层**：到访率是平时的 1.65 倍，高于健康层的 1.49 倍。大促对「刚错过一个使用周期」的用户尤其有效，
  这和报告 04「早期风险 × 高强度最值得召回」的判断一致——大促本身就是一次大规模召回；
- 加速流失层也被拉动了 1.4 倍，但基数低，到访率只有 8%。

## 3. 购买转化：大促日分层对比（11-17）

11-17 是唯一一天购买数据基本完整的大促日，和上周日（11-10）对比。分层取各自前一个周四的快照（11-14 / 11-07）；
「窗口内新到访」= 快照日之后才第一次出现的用户。

In [4]:
def buyers_by_stage(day, snap):
    return q(f"""
        WITH st AS (
            SELECT v.user_id,
                   lifecycle_stage(DATE '{snap}' - v.first_seen_date, DATE '{snap}' - v.event_date) AS stage
            FROM dws_user_visit_interval v
            WHERE v.event_date <= DATE '{snap}'
              AND (v.next_active_date IS NULL OR v.next_active_date > DATE '{snap}')
        )
        SELECT COALESCE(st.stage, 'new_in_window') AS stage, COUNT(*) AS dau,
               AVG(CASE WHEN u.orders > 0 THEN 1 ELSE 0 END) AS conversion, SUM(u.gmv) AS gmv
        FROM dws_user_daily u LEFT JOIN st USING (user_id)
        WHERE u.event_date = DATE '{day}'
        GROUP BY 1
    """).set_index("stage")

b17, b10 = buyers_by_stage("2019-11-17", "2019-11-14"), buyers_by_stage("2019-11-10", "2019-11-07")
order = ["new_in_window", "new", "healthy", "early_risk", "accelerating", "long_lapsed"]
names = {**STAGE_CN, "new_in_window": "窗口内新到访"}
buy = pd.DataFrame({
    "11-10 转化率": b10.conversion, "11-17 转化率": b17.conversion,
    "转化率倍数": b17.conversion / b10.conversion,
    "11-17 GMV": b17.gmv, "增量 GMV 占比": (b17.gmv - b10.gmv) / (b17.gmv - b10.gmv).sum(),
}).loc[order]
buy.index = buy.index.map(names)
print(f"11-17 GMV {b17.gmv.sum():,.0f}，上周日 {b10.gmv.sum():,.0f}，是 {b17.gmv.sum() / b10.gmv.sum():.1f} 倍")
buy.style.format({"11-10 转化率": "{:.1%}", "11-17 转化率": "{:.1%}", "转化率倍数": "{:.1f}×",
                  "11-17 GMV": "{:,.0f}", "增量 GMV 占比": "{:.0%}"})

11-17 GMV 57,517,324，上周日 6,699,038，是 8.6 倍


,11-10 转化率,11-17 转化率,转化率倍数,11-17 GMV,增量 GMV 占比
stage,,,,,
窗口内新到访,4.8%,22.3%,4.7×,"14,054,582",24%
新用户,7.5%,23.2%,3.1×,"4,538,237",8%
健康,7.3%,25.0%,3.4×,"25,622,283",45%
早期风险,6.2%,21.9%,3.6×,"6,326,425",11%
加速流失,6.1%,20.8%,3.4×,"5,032,408",9%
长期沉默,5.1%,20.4%,4.0×,"1,943,389",4%


- 大促当天**各层用户的转化率都提升了 3–5 倍**，从 5%–7% 升至 20%–25%；老用户中长期沉默层提升最多（4.0 倍），大促期间的新访客为 4.7 倍；
- 增量 GMV 里健康层约占 45%，四分之一来自大促期间才第一次出现的新用户，风险层（早期风险 + 加速流失）合计约 20%。
  GMV 增量并非集中于健康用户，而是各层大致同比例放大；
- 这一天 GMV 是上周日的 8.6 倍。三天合计 GMV 因为埋点缺失无法得到，只能说至少是这一天的量级。

## 4. 大促后留存

看大促三天到访过的人，接下来 7 天（11-18 ~ 11-24）有没有再来；对照组是 11-01 ~ 11-03 到访过的人在 11-04 ~ 11-10 有没有再来。

In [5]:
stick = pd.DataFrame({
    "大促到访者 7 天内再来": camp.visited_and_returned / camp.visited,
    "对照到访者 7 天内再来": base.visited_and_returned / base.visited,
})
stick["差"] = stick.iloc[:, 0] - stick.iloc[:, 1]
stick.index = stick.index.map(STAGE_CN)
new_ret = q("""
    WITH p AS (
        SELECT user_id,
               CASE WHEN first_seen_date >= '2019-11-15' THEN 'campaign' ELSE 'base' END AS grp
        FROM dws_user_profile
        WHERE first_seen_date BETWEEN '2019-11-15' AND '2019-11-17' OR first_seen_date BETWEEN '2019-11-01' AND '2019-11-03'
    ),
    r AS (
        SELECT DISTINCT u.user_id
        FROM dws_user_daily u JOIN p USING (user_id)
        WHERE (p.grp = 'campaign' AND u.event_date BETWEEN '2019-11-18' AND '2019-11-24')
           OR (p.grp = 'base'     AND u.event_date BETWEEN '2019-11-04' AND '2019-11-10')
    )
    SELECT grp, COUNT(*) AS users, COUNT(r.user_id) * 1.0 / COUNT(*) AS returned
    FROM p LEFT JOIN r USING (user_id) GROUP BY 1
""").set_index("grp")
stick.loc["三天内首次出现的新用户"] = [new_ret.loc["campaign", "returned"], new_ret.loc["base", "returned"],
                              new_ret.loc["campaign", "returned"] - new_ret.loc["base", "returned"]]
stick.style.format({"大促到访者 7 天内再来": "{:.1%}", "对照到访者 7 天内再来": "{:.1%}", "差": "{:+.1%}"})

,大促到访者 7 天内再来,对照到访者 7 天内再来,差
stage,,,
新用户,36.7%,44.1%,-7.4%
健康,54.0%,66.8%,-12.8%
早期风险,39.9%,49.3%,-9.4%
加速流失,34.6%,42.0%,-7.5%
三天内首次出现的新用户,25.9%,24.2%,+1.7%


- **老用户：大促带回的用户留存更弱。** 每一层大促到访者之后 7 天再来的比例都比平时低 7–13 个百分点。
  原因之一是大促额外带来的多为低活跃用户；另一部分来自大促后整体流量回落的季节性。
  两种原因指向同一结论：大促的召回效果是一次性的，与报告 03 中「大促一周后健康层大量流入早期风险层」一致；
- **新用户：大促带来的新客质量不低于平时**（7 天内回访 25.9% vs 24.2%），大促没有拉低新客质量；
- **建议在大促后第 6–8 天，对大促期间到访的老用户补充一次触达**，这是报告 04、05 召回方案的直接应用场景。

## 5. 需求透支检验

In [6]:
weeks = [("10-18", "10-24"), ("10-25", "10-31"), ("11-01", "11-07"), ("11-08", "11-14"), ("11-18", "11-24")]
rows = []
for a, b in weeks:
    x = d[(d.event_date >= f"2019-{a}") & (d.event_date <= f"2019-{b}")]
    rows.append({"周": f"{a} ~ {b}", "GMV": x.gmv.sum(), "日均 DAU": x.dau.mean(), "下单转化率": x.buyer_conversion.mean(), "客单价": x.gmv.sum() / x.orders.sum()})
wk = pd.DataFrame(rows)
wk["阶段"] = ["", "", "大促前第 2 周", "大促前 1 周", "大促后 1 周"]
wk.style.format({"GMV": "{:,.0f}", "日均 DAU": "{:,.0f}", "下单转化率": "{:.2%}", "客单价": "{:.0f}"}).hide(axis="index")

周,GMV,日均 DAU,下单转化率,客单价,阶段
10-18 ~ 10-24,"54,181,760","221,976",8.13%,366,
10-25 ~ 10-31,"47,435,474","200,841",7.87%,365,
11-01 ~ 11-07,"49,753,839","253,167",7.00%,344,大促前第 2 周
11-08 ~ 11-14,"48,940,100","285,587",6.00%,349,大促前 1 周
11-18 ~ 11-24,"49,475,665","275,822",6.38%,338,大促后 1 周


大促后一周 GMV 与大促前两周基本持平（均在 4,900 万左右），**没有出现补偿性下滑，未见需求透支**。
需求转移更多发生在大促**之前**：大促前一周流量更高、转化率却最低（报告 02 的「观望期」），部分需求在等待大促。

数据窗口只覆盖大促后两周，更长期的影响（如三个月内的复购）无法评估。

## 复盘结论

| 问题 | 结论 | 可信度 |
|---|---|---|
| 拉来了谁 | 增量到访六成来自健康层；相对拉动最大的是早期风险层（1.65×） | 高：只用到访数据 |
| 谁在买 | 各层转化率提升 3–5 倍，GMV 同比例放大，健康层约占增量的 45% | 中：只有 11-17 一天的完整购买 |
| 留下了吗 | 老用户留存比平时差，召回是一次性的；新客质量不差 | 中：混有大促后的季节性回落 |
| 透支了吗 | 大促后一周 GMV 没有下降；需求更多是「提前等」而不是「提前买」 | 中：只能看到两周 |
| GMV 多少 | 三天合计无法得到，11-17 单日是平时的 8.6 倍 | 低：埋点缺失 |

**下次大促前建议补充**：购买事件的小时级监控（本次中断 48 小时，到复盘时才被发现）、活动日历和渠道来源埋点、大促后第 6–8 天的定向触达。